# Phase 2B — Supplemental decision experiment

**Why:** instructor asks which reduction method and k to use, why truncated SVD beats full SVD, and the largest practical k where truncated SVD wins. This notebook uses the **real cached 20K × 384 MiniLM embeddings**, not random synthetic data. It compares **fit-only** times fairly, then evaluates **query latency and top-10 overlap** with the original embedding ranking. Results are saved only after execution.

**Note:** top-10 overlap is not relevance accuracy or recall against human labels. Query vectors are selected from the same corpus for reproducibility, with self-matches excluded. The 5 queries are a small diagnostic, not an unbiased held-out evaluation.

In [ ]:
import numpy as np, pandas as pd, time, gc
from pathlib import Path
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.preprocessing import normalize
from threadpoolctl import threadpool_limits
from google.colab import drive

drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/HP-SS-Engine')
OUT=ROOT/'phase2_matrix_ops'
DATA=OUT/'2A_sparse_dense'/'qwen_20k_minilm_384_float32.npy'
assert DATA.exists(),f'Missing cached embeddings: {DATA}'
X=np.load(DATA,mmap_mode='r')
assert X.shape==(20000,384) and X.dtype==np.float32
(OUT/'benchmarks').mkdir(parents=True,exist_ok=True)
(OUT/'figures').mkdir(parents=True,exist_ok=True)
print('Verified real embeddings:',X.shape)

## Configuration

First run **PILOT=True** (N=1000, k=50). After confirming the CSV, change to **PILOT=False** (N=1000/5000/20000 and k=50/100/200). All times use CPU, with BLAS limited to 2 threads for more reproducible comparison. Fit-only times exclude transformation, inverse transformation, and ranking. NumPy full SVD produces all components in one run; its measured fit time is repeated as a *reference threshold* across k, not falsely described as independent k-specific fits. Each full fit is timed **5 times** after **3 untimed warmups** for each size/method/k; pilot uses 3 timed repeats after 1 warmup. Retrieval timings use the same repeat/warmup policy. New outputs use `_full_5run` to preserve previous data.

In [ ]:
PILOT=True
SIZES=[1000] if PILOT else [1000,5000,20000]
KS=[50] if PILOT else [50,100,200]
REPEATS=3 if PILOT else 5
WARMUPS=1 if PILOT else 3
QUERY_IDS=[11,137,301,503,907]
TOP_K=10

# Fit-only comparison: identical input and BLAS thread limit.
fit_rows=[]
with threadpool_limits(limits=2):
    for n in SIZES:
        A=np.array(X[:n],dtype=np.float64,copy=True)
        # Warm up NumPy SVD on the SAME matrix shape; never include in timing.
        for _ in range(WARMUPS):
            np.linalg.svd(A,full_matrices=False)
        for rep in range(REPEATS):
            t0=time.perf_counter()
            U,S,Vt=np.linalg.svd(A,full_matrices=False)
            t_full=time.perf_counter()-t0
            for k in KS:
                fit_rows.append(dict(n_docs=n,k=k,method='numpy_full_svd',repeat=rep,fit_s=t_full))
            print('Full SVD',n,'rep',rep,'seconds',round(t_full,3),flush=True)
            del U,S,Vt
        for k in KS:
            # Warm up each fitted algorithm at each size and component count.
            for method in ['truncated_svd','pca']:
                for _ in range(WARMUPS):
                    warm_model=(TruncatedSVD(n_components=k,n_iter=7,random_state=42,algorithm='randomized')
                        if method=='truncated_svd' else PCA(n_components=k,svd_solver='randomized',iterated_power=7,random_state=42))
                    warm_model.fit(A)
            for rep in range(REPEATS):
                for method in ['truncated_svd','pca']:
                    if method=='truncated_svd':
                        model=TruncatedSVD(n_components=k,n_iter=7,random_state=42,algorithm='randomized')
                    else:
                        model=PCA(n_components=k,svd_solver='randomized',iterated_power=7,random_state=42)
                    t0=time.perf_counter()
                    model.fit(A)
                    elapsed=time.perf_counter()-t0
                    fit_rows.append(dict(n_docs=n,k=k,method=method,repeat=rep,fit_s=elapsed))
                    print(n,k,method,'rep',rep,round(elapsed,3),'s',flush=True)
        del A;gc.collect()
fit_df=pd.DataFrame(fit_rows)
suffix='pilot_5run' if PILOT else 'full_5run'
fit_path=OUT/'benchmarks'/f'phase2b_fit_only_{suffix}.csv'
fit_df.to_csv(fit_path,index=False)
print('Saved',fit_path)
print(fit_df.groupby(['n_docs','k','method']).fit_s.median().unstack().round(3))

## Retrieval diagnostic: normalized cosine and top-10 overlap

At each corpus size, train TruncatedSVD and PCA on the same real vectors; use `transform` on corpus and queries. L2-normalize **after** reduction, because cosine requires normalized vectors. Baseline ranks the original 384-dimensional embeddings. Exclude the query document itself in both rankings. Timing is **ranking only** (matrix-vector dot and top-k selection), after fitting and transformation. Run 5 queries × 5 repeats per condition. We record overlap@10 and speed relative to original baseline; this is *not* ground-truth recall@10.

In [ ]:
def topk_cosine(corpus,query,k,excluded_index):
    scores=corpus@query
    scores[excluded_index]=-np.inf
    candidates=np.argpartition(scores,-k)[-k:]
    return candidates[np.argsort(scores[candidates])[::-1]]

search_rows=[]
with threadpool_limits(limits=2):
    for n in SIZES:
        A=np.asarray(X[:n],dtype=np.float32)
        baseline=normalize(A,axis=1,copy=True).astype(np.float32)
        for k in KS:
            reduced={'original_384':baseline}
            for method in ['truncated_svd','pca']:
                if method=='truncated_svd':
                    model=TruncatedSVD(n_components=k,n_iter=7,random_state=42)
                else:
                    model=PCA(n_components=k,svd_solver='randomized',iterated_power=7,random_state=42)
                Z=model.fit_transform(A)
                reduced[method]=normalize(Z,axis=1).astype(np.float32)
            for qid in QUERY_IDS:
                baseline_top=topk_cosine(baseline,baseline[qid],TOP_K,qid)
                for method,Z in reduced.items():
                    for _ in range(WARMUPS):
                        topk_cosine(Z,Z[qid],TOP_K,qid)
                    for rep in range(REPEATS):
                        t0=time.perf_counter()
                        selected=topk_cosine(Z,Z[qid],TOP_K,qid)
                        elapsed_ms=(time.perf_counter()-t0)*1000
                        overlap=len(set(baseline_top.tolist()) & set(selected.tolist()))/TOP_K
                        search_rows.append(dict(n_docs=n,k=k,method=method,query_id=qid,repeat=rep,search_ms=elapsed_ms,top10_overlap=overlap))
            print('Search finished',n,k,flush=True)
search_df=pd.DataFrame(search_rows)
search_path=OUT/'benchmarks'/f'phase2b_retrieval_{suffix}.csv'
search_df.to_csv(search_path,index=False)
print('Saved',search_path)
print(search_df.groupby(['n_docs','k','method'])[['search_ms','top10_overlap']].median().round(3))

## Summary figure and cautious decision aid

The plot uses **measured values only**. The practical maximum tested k where TruncatedSVD beats full SVD is defined **per N**, using median fit-only time; do not extrapolate beyond k=200. Full SVD is a one-off decomposition whose cost does not change with requested truncation k. If randomized SVD never wins at a size, report *none*.

In [ ]:
import matplotlib.pyplot as plt
med=fit_df.groupby(['n_docs','k','method'],as_index=False).fit_s.median()
fig,axes=plt.subplots(1,3,figsize=(17,4.5))
for method,g in med[med.n_docs==max(SIZES)].groupby('method'):
    axes[0].plot(g.k,g.fit_s,'o-',label=method)
axes[0].set(xlabel='Components k',ylabel='Fit-only seconds',title=f'Fit-only at N={max(SIZES)}')
for method,g in med[med.k==min(KS)].groupby('method'):
    axes[1].plot(g.n_docs,g.fit_s,'o-',label=method)
axes[1].set(xlabel='Documents N',ylabel='Fit-only seconds',title=f'Scaling at k={min(KS)}')
sel=search_df[search_df.n_docs==max(SIZES)].groupby(['k','method'],as_index=False).agg(search_ms=('search_ms','median'),top10_overlap=('top10_overlap','mean'))
for method,g in sel.groupby('method'):
    if method!='original_384':axes[2].plot(g.k,g.top10_overlap,'o-',label=method)
axes[2].set(xlabel='Components k',ylabel='Mean top-10 overlap',ylim=(0,1.05),title=f'Ranking agreement at N={max(SIZES)}')
for ax in axes:ax.grid(alpha=.25);ax.legend(fontsize=8)
fig.tight_layout();path=OUT/'figures'/f'phase2b_decision_{suffix}.png'
fig.savefig(path,dpi=150,bbox_inches='tight');plt.show();print('Saved',path)
for n in SIZES:
    pivot=med[med.n_docs==n].pivot(index='k',columns='method',values='fit_s')
    wins=pivot.index[pivot['truncated_svd']<pivot['numpy_full_svd']].tolist()
    print('N=',n,'largest tested winning k=',max(wins) if wins else 'none','(tested only)',KS)
print('Reminder: overlap is NOT human-rated retrieval accuracy; results depend on query selection and CPU load.')

## What to write in the report

- Full vs truncated: compare **fit-only** median times at matched N and thread count; state the largest *tested* k with advantage.
- PCA differs from TruncatedSVD because PCA centers the columns.
- Choose k based on measured query latency, storage, top-10 agreement, and reconstruction error from notebook 07.
- Limitations: 5 in-corpus query documents, no independent relevance labels, single CPU environment, and ranking timings exclude embedding/model inference and fit cost.
- Never claim top-10 overlap equals semantic accuracy or recall against relevance judgments.